# Walidacja datasetu (shardy) – skrócona dokumentacja

## 1. Cel
Szybkie potwierdzenie, że świeżo wygenerowany zbiór pozycji (shardy CSV) jest jakościowo stabilny przed etapami: feature engineering, baseline training, budowa „gold subset”.

---

## 2. Wejście
- Folder z plikami `positions_shard_*.csv`
- Kolumny m.in.: `fen, eval_cp, phase_bucket, time_ms, depth, material_white, material_black`
- Parametry stałe w kodzie: ścieżki, limity próbek, CLAMP_CP.

---


## 3. Główne zadania skryptu
1. Iteracyjne wczytanie shardów chunkami (stream, niskie zużycie RAM).
2. Agregacja liczebności faz: OPEN / MID / END.
3. Zliczenie pozycji z wartością przyciętą (|eval_cp| == CLAMP_CP).
4. Zebranie ograniczonych próbek do histogramów: eval_cp, time_ms, depth, różnica materiału.
5. Filtrowanie depth == -1 z wizualizacji (oddzielne raportowanie odsetka).
6. Generacja wykresów PNG + tekstowy `summary.txt` ze statystykami.

---


## 4. Dlaczego te metryki
- Phase distribution – wykrywa bias (nadreprezentacja jednej fazy może zniekształcić trening).
- clamp_ratio – sygnał, czy clamp „ucina” ogon rozkładu (utrata informacji o dużych przewagach).
- Percentyle eval (p50/p90/p99) – kształt rozkładu, outliery, symetria.
- Time_ms – stabilność pracy silnika (szczególnie p90/p99).
- Depth – realna głębokość fast eval (czy nie jest zbyt płytko).
- Material diff – czy miks pozycji obejmuje zróżnicowane stany (nie tylko końcówki / nie tylko otwarcia).

---


## 5. Interpretacja przykładowych wyników
- Fazy: 30% / 30% / 40% (END nadbity – OK jeśli planowany endgame focus; inaczej warto obniżyć sampling końcówek).
- clamp_ratio ≈ 0.15% (bardzo niskie → clamp nie ogranicza).
- Eval p99 = 552 cp << 2000 (duży margines, clamp niewykorzystany – można zostawić).
- Czas: mean ~15 ms, p99 ~48 ms (stabilne, brak ekstremów).
- Depth: mean ≈10, p99 = 14 (typowa szybka analiza; wymaga później głębszego „gold subset” do weryfikacji).
- Ogólny wniosek: dane nadają się do startu baseline’u.

---


## 6. Progi orientacyjne (szybka heurystyka)
- clamp_ratio: OK <3% | ostrzeżenie 3–6% | alarm >6%.
- Udział END (jeśli celem balans): OK ≤38–40% | ostrzeżenie 40–45% | alarm >45%.
- depth -1: OK <0.5% | ostrzeżenie 0.5–1% | alarm >1%.
- Eval p99 blisko CLAMP_CP + rosnący clamp_ratio → rozważyć wyższy clamp lub dodatkową flagę saturacji.

---


## 7. Co świadomie pominięto
- Sprawdzenie legalności FEN.
- Kontrola duplikatów (brak setu FEN w tej wersji).
- Walidacja `eval_norm`.
- Analiza korelacji depth ↔ time ↔ eval.
- Segmentacja per side_to_move.

---


## 8. Potencjalne ryzyka i konsekwencje
1. Nierównowaga faz → model preferuje najczęstszą fazę.
2. Wysoki clamp_ratio → spłaszczony target, gorsza nauka na dużych przewagach.
3. Zbyt płytka depth → większy szum w targetach (niższy górny sufit jakości).
4. Niestabilne czasy (duży ogon time_ms) → nierówna jakość eval, trudniejsza interpretacja błędów.
5. Skośny material diff → model uczy się głównie specyficznej klasy pozycji.

---


## 9. Minimalne kolejne kroki
1. Dodać walidację `eval_norm` (różnica < 1e-3 względem `eval_cp/CLAMP_CP`).
2. Policzyć unikalne reduced FEN i raportować duplikaty.
3. Wygenerować manifest JSON (per shard: count, min/max eval, fazy).
4. Przygotować baseline MLP i raport MAE_cp vs percentyle eval.
5. Utworzyć „gold subset” z większą głębokością / MultiPV do oceny systematycznego biasu.

---


## 10. Krótki wniosek
Snapshot wygląda zdrowo (niska saturacja, stabilne czasy, umiarkowana głębokość, kontrolowane przesunięcie faz). Można przejść do fazy modelowania, jednocześnie rozszerzając walidację o duplikaty i eval_norm.

---


## 11. Wykresy
![Phase Distribution](../../plots/check_data_imbalance/phase_distribution.png)
![Eval Histogram](../../plots/check_data_imbalance/eval_hist.png)
![Eval CDF](../../plots/check_data_imbalance/eval_cdf.png)
![Time Histogram](../../plots/check_data_imbalance/time_hist.png)
![Time Log Histogram](../../plots/check_data_imbalance/time_hist_log.png)
![Depth Histogram](../../plots/check_data_imbalance/depth_hist.png)
![Material Difference Histogram](../../plots/check_data_imbalance/material_diff_hist.png)